# 03 · The Same Agent via LangChain / LangGraph

**Where we are in the stack:**
- `ChatOpenAI` (LangChain model layer) = the **HAL / SAI** - swap providers by changing one line.
- `create_react_agent` (LangGraph) = the **prebuilt control-plane FSM** - your **FRR**.

Same two tools. Same loop. The 35-line loop from notebook 2 collapses to **one line** - and you get retries, parallel calls, streaming, memory and checkpointing for free.

In [ ]:
%pip install -q langchain langgraph langchain-openai

## 1. The model layer = the HAL

One object, provider-agnostic. To move from OpenAI to a local model or to Anthropic, you change the class/args here and nothing else downstream - exactly like SAI letting one NOS run on Broadcom or Marvell.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


# Behind a TLS-intercepting firewall/proxy, HTTPS cert verification can fail.
# Set VERIFY_SSL=false in .env to skip it: we hand ChatOpenAI custom httpx
# clients with verification turned off. Leave it true everywhere else.
import httpx
VERIFY_SSL = os.environ.get("VERIFY_SSL", "true").strip().lower() not in ("false", "0", "no")
if not VERIFY_SSL:
    import warnings
    warnings.filterwarnings("ignore")
    print("\u26a0\ufe0f  SSL verification DISABLED (VERIFY_SSL=false) \u2014 use only on a trusted network")

model = ChatOpenAI(
    model=os.environ.get("MODEL", "gpt-4o-mini"),
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ.get("OPENAI_API_KEY", "set-me"),
    temperature=0,
    http_client=httpx.Client(verify=VERIFY_SSL),
    http_async_client=httpx.AsyncClient(verify=VERIFY_SSL),
)

# Provider swap (the HAL point) - uncomment to run on Anthropic instead:
# from langchain_anthropic import ChatAnthropic
# model = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

## 2. Tools via the `@tool` decorator

The type hints + docstring auto-generate the JSON schema you wrote by hand in notebook 2.

In [ ]:
import ipaddress, hashlib
from langchain_core.tools import tool

@tool
def calculate_subnet(cidr: str) -> dict:
    '''Compute network, broadcast, netmask and usable host count for a CIDR (e.g. 10.20.0.0/22).'''
    net = ipaddress.ip_network(cidr, strict=False)
    usable = (net.num_addresses - 2 if net.version == 4 and net.prefixlen <= 30
              else net.num_addresses)
    return {"network": str(net.network_address),
            "broadcast": str(net.broadcast_address) if net.version == 4 else "n/a",
            "netmask": str(net.netmask), "prefix_length": net.prefixlen,
            "total_addresses": net.num_addresses, "usable_hosts": usable}

@tool
def get_interface_status(device: str, interface: str) -> dict:
    '''Operational status and error counters for an interface on a device (mocked telemetry).'''
    h = int(hashlib.md5(f"{device}{interface}".encode()).hexdigest(), 16)
    return {"device": device, "interface": interface, "admin_status": "up",
            "oper_status": "up" if h % 5 != 0 else "down", "speed": "10Gbps",
            "mtu": 1500, "input_errors": h % 7, "output_errors": h % 3, "crc_errors": h % 4}

tools = [calculate_subnet, get_interface_status]

## 3. The agent = one line (LangGraph builds the FSM for you)

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(model, tools)   # this is your whole control loop

## 4. Run it

In [ ]:
result = agent.invoke({"messages": [
    ("user", "Is ethernet1/0/1 on leaf-01 up, and how many usable hosts are in 10.20.0.0/22?")
]})
print(result["messages"][-1].content)

## 5. Observability - stream the FSM transitions

`.stream()` exposes every intermediate step (model turn, tool call, tool result). This is exactly the hook point you'd send to Langfuse for cost/latency/groundedness.

In [ ]:
for chunk in agent.stream(
    {"messages": [("user",
        "Check whether et-0/0/1 on spine-02 is up; if up, give the broadcast of 192.168.10.0/24.")]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

## What the framework gave you for free

The 35-line loop in notebook 2 still exists - it's just inside `create_react_agent` now. On top of it you also got: retries, parallel tool calls, streaming, persistent memory + checkpointing (`MemorySaver`), and a typed state graph you can extend with custom nodes / confirmation gates.

That is the FRR argument: you *could* hand-roll BGP, but you run FRR so you inherit the hard parts.

## The whole arc, mapped to the diagram

| Notebook | Stack layer | Networking analog |
|---|---|---|
| 01 chat completions | LLM + Provider SDK | ASIC + Broadcom SDK (over HTTPS) |
| 02 agent from scratch | Agent (control plane) | hand-rolled BGP state machine |
| 03 langchain/langgraph | model layer + agent framework | SAI (HAL) + FRR |

A UI (ChatGPT/Cursor) would sit on top as the **management plane** - and bundles all of the above into one box.